# Listing Segmentation — Lima, Perú

K-means clustering on the gold layer.
Business question: ¿qué segmentos de listings existen en Lima por perfil de calidad, precio y disponibilidad?

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', palette='muted')

## 1. Load gold segments

In [ ]:
df = pd.read_parquet('../data/gold/mart_listing_segments.parquet')
print(f'Rows: {len(df):,} | Segments: {df["segment"].nunique()}')
df['segment'].value_counts().sort_index()

## 2. Silhouette scan (re-run for reference)

The pipeline already chose the optimal k. This cell documents the scores.

In [ ]:
profiles = pd.read_csv('../data/gold/segment_profiles.csv', index_col='segment')
profiles

## 3. Cluster size

In [ ]:
seg_counts = df['segment'].value_counts().sort_index()
seg_counts.plot(kind='bar', figsize=(7, 4), title='Listings per segment')
plt.xlabel('Segment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4. PCA 2D visualization

In [ ]:
import yaml
with open('../configs/lima_schema.yaml') as f:
    schema = yaml.safe_load(f)
seg_cfg = schema['segmentation']

from sklearn.preprocessing import StandardScaler, OneHotEncoder

log1p_cols = seg_cfg['features']['log1p_scaled']
scale_cols = seg_cfg['features']['scaled']
onehot_cols = seg_cfg['features']['onehot']

working = df[log1p_cols + scale_cols + onehot_cols + ['segment']].dropna()

log1p_scaled = StandardScaler().fit_transform(np.log1p(working[log1p_cols].values))
scaled = StandardScaler().fit_transform(working[scale_cols].values)
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe = enc.fit_transform(working[onehot_cols].values)
X = np.hstack([log1p_scaled, scaled, ohe])

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)
print(f'Explained variance: {pca.explained_variance_ratio_.sum():.1%}')

pca_df = pd.DataFrame(coords, columns=['PC1', 'PC2'])
pca_df['segment'] = working['segment'].values

plt.figure(figsize=(9, 6))
for seg, grp in pca_df.groupby('segment'):
    plt.scatter(grp['PC1'], grp['PC2'], label=f'Segment {seg}', alpha=0.4, s=10)
plt.title('K-means segments — PCA 2D projection')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Segment profiles — median features

In [ ]:
feat_cols = log1p_cols + scale_cols
profile_detail = working.groupby('segment')[feat_cols].median().round(2)
profile_detail

## 6. Radar / bar profile per segment

In [ ]:
from sklearn.preprocessing import MinMaxScaler

norm = MinMaxScaler().fit_transform(profile_detail.T)
norm_df = pd.DataFrame(norm, index=profile_detail.columns, columns=profile_detail.index)

norm_df.T.plot(kind='bar', figsize=(11, 5), title='Normalized feature medians per segment')
plt.xticks(rotation=0)
plt.ylabel('Normalized value (0-1)')
plt.tight_layout()
plt.show()

## 7. Business interpretation

Label each segment based on the profile above.

| Segment | Label | Description | Business action |
|---|---|---|---|
| 0 | _(fill after analysis)_ | _(fill)_ | _(fill)_ |
| 1 | _(fill after analysis)_ | _(fill)_ | _(fill)_ |
| 2 | _(fill after analysis)_ | _(fill)_ | _(fill)_ |

Example interpretations:
- **Alta disponibilidad, bajo precio, pocas reseñas** → listings nuevos o inactivos — riesgo de calidad, candidatos a onboarding.
- **Alta calificación, precio medio, buena disponibilidad** → ancla de confianza del marketplace.
- **Precio alto, pocas noches mínimas, alta demanda** → listings premium — foco en retención de host.

## 8. Room type breakdown per segment

In [ ]:
ct = working.groupby(['segment', 'room_type']).size().unstack(fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot(kind='bar', stacked=True, figsize=(9, 5), title='Room type mix per segment (%)')
plt.xticks(rotation=0)
plt.ylabel('%')
plt.tight_layout()
plt.show()